# Lesson 03 - Agentic Disaini Musterid

Selles õppetükis uurime kolme põhilist disainimustrit tõhusate tehisintellekti agentide loomisel:

1. **Selged Agendi Juhised** — Täpsete, rolli määratlevate käskude koostamine, mis juhivad agendi käitumist
2. **Struktureeritud Väljund Pydantic Mudelitega** — Tagamaks, et agendid tagastavad ennustatavaid ja valideeritud andmeid
3. **Ühe Vastutusega Agendid** — Keskendunud agentide kavandamine, kes teevad igaüks üht asja hästi

Rakendame iga mustri **reisisihtkohtade soovitajate** stsenaariumis, ehitades järk-järgult süsteemi, mis suudab soovitada sihtkohti, kontrollida saadavust ja korraldada logistikat.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Muster 1: Selged Agendi Juhised

Kõige mõjukam muster on ka kõige lihtsam: kirjutada oma agendile selged, üksikasjalikud juhised.

Head juhised määratlevad:
- **Kes** agent on (persoon ja toon)
- **Mida** ta peaks tegema (samm-sammult ülesanded)
- **Kuidas** ta peaks käituma (piirangud ja stiil)

Allpool loome reisikonsjerži agendi, kellel on sõnaselged juhised, mis kujundavad iga tema vastuse.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Struktureeritud väljund Pydantic mudelitega

Vaba tekst on vestluses kasulik, kuid allavoolu süsteemid vajavad struktureeritud andmeid.
Paaristades **Pydantic mudelid** ja **tööriista funktsiooni**, saame:

- Määratleda täpse skeemi agendi väljundi jaoks
- Vastuseid automaatselt valideerida
- Integreerida agendi tulemused rakenduse loogikasse usaldusväärselt

Tutvustame ka tööriista, mis tagastab sihtkoha andmed, et agent saaks oma soovitused põhineda reaalsetel andmetel.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Muster 3: Ühe vastutusega agendid

Komplekssed ülesanded saavad kasu töölaiendamisest mitme keskendunud agendi vahel, kellel kõigil on üksainus vastutusala:

- **Sihtkoha ekspert**, kes tunneb kohti ja saadavust
- **Logistika planeerija**, kes tegeleb lendude, hotellide ja marsruutidega

See peegeldab tarkvaraarenduse põhimõtet *hoolikuse eraldamine* — iga agent on lihtsam iseseisvalt testida, hooldada ja täiustada.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Kokkuvõte

Selles õppetükis rakendasime kolme agentlikku disainimustrit reisisoovituse stsenaariumi jaoks:

| Muster | Peamine idee | Kasu |
|---|---|---|
| **Selged juhised** | Määra alguses persoon, vastutused ja piirangud | Järjepidev, kaubamärgile vastav agendi käitumine |
| **Struktureeritud väljund** | Kasuta vastuse vorminguna Pydanticu mudeleid | Kinnitatud, masinloetavad tulemused |
| **Üks vastutusala** | Anna igale agendile üks keskendunud ülesanne | Kergem testida, hooldada ja kombineerida |

Need mustrid sobituvad loomulikult kokku — saad kombineerida selged juhised struktureeritud väljundiga ühe vastutusalaga agendi sees, et luua vastupidavaid, tootmiskõlblikke süsteeme.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutülesanne**:
See dokument on tõlgitud kasutades tehisintellekti tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame tagada täpsust, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Tähtsa teabe puhul soovitatakse professionaalset inimtõlget. Me ei vastuta ühegi arusaamatusi ega väärarusaamu eest, mis võivad tekkida selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
